In [ ]:
!pip install ultralytics -q

In [ ]:
from ultralytics import YOLO
import os
import shutil
from PIL import Image

DATASET_PATH = "/kaggle/input/datasets/kushagrapandya/visdrone-dataset"

os.makedirs("visdrone/images/train", exist_ok=True)
os.makedirs("visdrone/images/val", exist_ok=True)
os.makedirs("visdrone/labels/train", exist_ok=True)
os.makedirs("visdrone/labels/val", exist_ok=True)

In [ ]:
def convert_visdrone_to_yolo(label_path, save_path, img_width, img_height):

    with open(label_path, 'r') as f:
        lines = f.readlines()

    yolo_lines = []

    for line in lines:
        parts = line.strip().split(',')

        if len(parts) < 6:
            continue

        x, y, w, h, score, class_id = map(int, parts[:6])

        if class_id != 1:   # keep PERSON only
            continue

        x_center = (x + w/2) / img_width
        y_center = (y + h/2) / img_height
        w_norm = w / img_width
        h_norm = h / img_height   # ✅ THIS IS THE FIX

        yolo_lines.append(f"0 {x_center} {y_center} {w_norm} {h_norm}\n")

    with open(save_path, 'w') as f:
        f.writelines(yolo_lines)

In [ ]:
def process_split(image_dir, label_dir, out_img_dir, out_lbl_dir):

    for img_name in os.listdir(image_dir):

        img_path = os.path.join(image_dir, img_name)
        label_name = img_name.replace(".jpg",".txt")
        label_path = os.path.join(label_dir, label_name)

        if not os.path.exists(label_path):
            continue

        img = Image.open(img_path)
        w, h = img.size

        shutil.copy(img_path, os.path.join(out_img_dir, img_name))

        convert_visdrone_to_yolo(
            label_path,
            os.path.join(out_lbl_dir, label_name),
            w,
            h
        )

In [ ]:
process_split(
    f"{DATASET_PATH}/VisDrone2019-DET-train/VisDrone2019-DET-train/images",
    f"{DATASET_PATH}/VisDrone2019-DET-train/VisDrone2019-DET-train/annotations",
    "visdrone/images/train",
    "visdrone/labels/train"
)

In [ ]:
process_split(
    f"{DATASET_PATH}/VisDrone2019-DET-val/VisDrone2019-DET-val/images",
    f"{DATASET_PATH}/VisDrone2019-DET-val/VisDrone2019-DET-val/annotations",
    "visdrone/images/val",
    "visdrone/labels/val"
)

In [ ]:
with open("visdrone.yaml", "w") as f:
    f.write("""
path: visdrone
train: images/train
val: images/val

names:
  0: person
""")

In [ ]:
model = YOLO("yolov8s.pt")

model.train(
    data="visdrone.yaml",
    epochs=30,
    imgsz=1280,
    batch=8,
    optimizer="AdamW",
    lr0=0.0008,
    mosaic=1.0,
    mixup=0.1,
    degrees=5,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    device=0,
    name="final_drone_model"
)

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=visdrone.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0008, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=final_drone_model2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=100, perspective=0.0, 

In [ ]:
model = YOLO("/kaggle/working/runs/detect/final_drone_model2/weights/best.pt")

In [ ]:
metrics = model.val()

print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP@50:", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1939.0±1089.7 MB/s, size: 98.7 KB)
val: Scanning /kaggle/working/visdrone/labels/val.cache... 548 images, 28 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 121.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 2.0it/s 17.4s0.5s
                   all        548       8844      0.728      0.617      0.689      0.335
Speed: 2.0ms preprocess, 23.0ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to /kaggle/working/runs/detect/val2
Precision: 0.7275805956042248
Recall: 0.6168023518769787
mAP@50: 0.6894396446756718
mAP@50-95: 0.335178937864247


In [ ]:
model.export(format="onnx")

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/kaggle/working/runs/detect/final_drone_model2/weights/best.pt' with input shape (1, 3, 1280, 1280) BCHW and output shape(s) (1, 5, 33600) (21.5 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 282ms
Prepared 2 packages in 2.94s
Installed 2 packages in 9ms
 + onnxruntime-gpu==1.24.2
 + onnxslim==0.1.86

requirements: AutoUpdate success ✅ 3.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 22...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1447: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  warnings.warn(


ONNX: slimming with onnxslim 0.1.86...
ONNX: export success ✅ 6.6s, saved as '/kaggle/working/runs/detect/final_drone_model2/weights/best.onnx' (43.2 MB)

Export complete (9.2s)
Results saved to /kaggle/working/runs/detect/final_drone_model2/weights
Predict:         yolo predict task=detect model=/kaggle/working/runs/detect/final_drone_model2/weights/best.onnx imgsz=1280 
Validate:        yolo val task=detect model=/kaggle/working/runs/detect/final_drone_model2/weights/best.onnx imgsz=1280 data=visdrone.yaml  
Visualize:       https://netron.app


'/kaggle/working/runs/detect/final_drone_model2/weights/best.onnx'

In [ ]:
#create drone test video
import os

img_folder = "/kaggle/input/datasets/kushagrapandya/visdrone-dataset/VisDrone2019-DET-test-dev/VisDrone2019-DET-test-dev/images"

images = sorted(os.listdir(img_folder))

VIDEO_WIDTH = 1280
VIDEO_HEIGHT = 720

out = cv2.VideoWriter(
    "drone_test.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    10,
    (VIDEO_WIDTH, VIDEO_HEIGHT)
)

for image in images[:200]:

    img = cv2.imread(os.path.join(img_folder, image))

    if img is None:
        continue

    img = cv2.resize(img, (VIDEO_WIDTH, VIDEO_HEIGHT))   # 🔥 FIX HERE

    out.write(img)

out.release()

print("Video created successfully!")

Video created successfully!


In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import time
from IPython.display import Video

model = YOLO("/kaggle/working/runs/detect/final_drone_model2/weights/best.pt")

cap = cv2.VideoCapture("drone_test.mp4")

GRID_ROWS = 6
GRID_COLS = 6

T1 = 2
T2 = 4

VIDEO_WIDTH = 1280
VIDEO_HEIGHT = 720

out = cv2.VideoWriter(
    "crowd_output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    10,
    (VIDEO_WIDTH, VIDEO_HEIGHT)
)

frame_count = 0
start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (VIDEO_WIDTH, VIDEO_HEIGHT))

    # Detect persons
    results = model.predict(frame, imgsz=640, conf=0.3, verbose=False)
    boxes = results[0].boxes.xyxy.cpu().numpy()

    h, w = frame.shape[:2]

    cell_w = w // GRID_COLS
    cell_h = h // GRID_ROWS
    cell_area = cell_w * cell_h     # A in the formula

    density_map = np.zeros((GRID_ROWS, GRID_COLS))

    # Count detections per cell (N)
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)

        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)

        gx = cx // cell_w
        gy = cy // cell_h

        if gx < GRID_COLS and gy < GRID_ROWS:
            density_map[gy, gx] += 1

    # Compute Density + Risk
    for i in range(GRID_ROWS):
        for j in range(GRID_COLS):

            N = density_map[i, j]      # detections
            D = N         # D = N / A  ✅

            # Risk Classification (from slide)
            if D < T1:
                color = (0,255,0)      # Low Risk (Green)
                label = "LOW"
            elif T1 <= D < T2:
                color = (0,255,255)    # Medium Risk (Yellow)
                label = "MED"
            else:
                color = (0,0,255)      # High Risk (Red)
                label = "HIGH"

            x1 = j * cell_w
            y1 = i * cell_h
            x2 = x1 + cell_w
            y2 = y1 + cell_h

            cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
            cv2.putText(frame,label,(x1+5,y1+20),
                        cv2.FONT_HERSHEY_SIMPLEX,0.6,color,2)

    total_people = len(boxes)

    frame_count += 1
    fps = frame_count / (time.time() - start_time)

    cv2.putText(frame,f"People: {total_people}",(20,40),
                cv2.FONT_HERSHEY_SIMPLEX,1,(255,0,0),2)

    cv2.putText(frame,f"FPS: {fps:.2f}",(20,80),
                cv2.FONT_HERSHEY_SIMPLEX,1,(255,255,0),2)

    out.write(frame)

cap.release()
out.release()

print("Processing Done!")
print("Average FPS:", fps)

Processing Done!
Average FPS: 34.874693018516005


In [ ]:
Video("crowd_output.mp4")

In [ ]:
from google.colab import files
files.upload()

Saving best.pt to best.pt
Buffered data was truncated after reaching the output size limit.

In [ ]:
model = YOLO("best.pt")